In [ ]:
import pandas as pd

cs_df = pd.read_csv('customer_satisfaction.csv')
ct_df = pd.read_csv('customer_transactions.csv')
mp_df = pd.read_csv('marketing_performance.csv')

cs_df.head()
# ct_df.head(20)
# mp_df.head()



,customer_id,contact_date,contact_reason,satisfaction_score,resolution_time_hours,repeat_contact
0,495,2023-06-16,technical,4.0,11.4,False
1,495,2024-03-21,shipping,1.8,8.1,True
2,727,2023-07-03,product,4.3,4.0,False
3,86,2023-07-04,technical,4.3,2.0,False
4,471,2023-08-17,technical,4.2,5.0,False


문제 1: 채널별 고객 획득 비용(CAC) 및 생애 가치(LTV) 분석
목표: 마케팅 채널의 진정한 ROI를 평가하기 위해 CAC와 LTV를 계산하고 최적 투자 전략을 제안하세요.
1. 데이터 로드 및 전처리
2. 채널별 CAC 계산
    - 2024년 데이터만 사용
    - 각 채널별 총 마케팅 비용 집계
    - 각 채널별 신규 고객 수 집계 (2024년 첫 거래 기준)
    - CAC = 총 마케팅 비용 / 신규 고객 수
    - organic과 referral 채널은 CAC = 0으로 처리

In [66]:
mp_df['year'] = pd.to_datetime(mp_df['month'])
mp_df_2024 = mp_df[mp_df['year'].dt.year == 2024]

# 각 채널별 총 마케팅 비용 집계
total_2024 = mp_df_2024.groupby('channel')['marketing_spend'].sum().reset_index()
total_2024

# 각 채널별 신규 고객 수 집계 (2024년 첫 거래 기준)
new_C_2024 = mp_df_2024.groupby('channel')['new_customers'].count()
new_C_2024

# CAC = 총 마케팅 비용 / 신규 고객 수
mp_df_2024['cac'] = mp_df_2024['marketing_spend'].sum() / mp_df_2024['new_customers'].sum() 
mp_df_2024.head()

g_cac_2024 = mp_df_2024.groupby('channel')['marketing_spend'].sum() / mp_df_2024.groupby('channel')['new_customers'].sum() 
g_cac_2024
# organic과 referral 채널은 CAC = 0으로 처리
g_cac_2024[['organic', 'referral']] = 0.0
g_cac_2024


C:\Users\mumu1\AppData\Local\Temp\ipykernel_51076\1524129651.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mp_df_2024['cac'] = mp_df_2024['marketing_spend'].sum() / mp_df_2024['new_customers'].sum()


channel
email           112.425752
organic           0.000000
paid_search     436.826024
referral          0.000000
social_media    327.630092
dtype: float64

In [91]:
ct_df.head()

,customer_id,registration_date,acquisition_channel,customer_segment,transaction_date,order_value,product_category
0,1,2023-04-13,email,premium,2023-04-15,104.24,Books
1,1,2023-04-13,email,premium,2023-11-16,141.21,Electronics
2,1,2023-04-13,email,premium,2023-11-19,129.03,Books
3,1,2023-04-13,email,premium,2023-08-29,67.59,Home
4,1,2023-04-13,email,premium,2023-10-08,102.30,Books


In [ ]:
# 3. 고객별 LTV 계산
ct_df['registration_date'] = pd.to_datetime(ct_df['registration_date'])
ct_df['transaction_date'] = pd.to_datetime(ct_df['transaction_date'])

ct_df['order_value'] = ct_df[ct_df['order_value'].dt.year == 2024]
ct_df['customer_id'] = ct_df[ct_df['customer_id'].dt.year == 2024]

# rdate_2024 = ct_df[ct_df['registration_date'].dt.year == 2024]
# tdate_2024 = ct_df[ct_df['transaction_date'].dt.year == 2024]

# ltv_2024 = ct_df['order_value'].mean() * ct_df['customer_id'].count() * (ct_df['transaction_date'] - ct_df['2024_registration_date'])

4. 채널별 LTV 및 ROI 분석 (`24년 신규 고객들의 평균 LTV / 24년 CAC`)   
    - 채널별 평균 LTV 계산
    - ROI = LTV / CAC 계산 (organic, referral은 무한대 처리)
    - payback period = CAC / (월평균 구매금액) 계산

5. 시각화 및 전략 제안 (15분)
    - CAC vs LTV 산점도 차트
    - 채널별 ROI 막대 차트
    - 마케팅 예산 재배분 제안 (현재 vs 최적 배분)